# Part 2: Entities and graphs

A Pantry Lane customer has a chocolate wrapper with batch code **EC26-0811** and wants to know which farms grew its cacao.

<img src="cocoarynth_bar_back.jpg" alt="The back of a Cocoarynth organic dark chocolate wrapper, with batch code EC26-0811 printed in the bottom left corner" width="400">

A single chunk can mention a batch or a farm, but a graph makes the connection explicit:

**Batch → BeanLot ← Farm**

**🎯 Mission**
- Compare a source document with a stored graph **assertion** (a subject, predicate, and object).
- Write a **GQL** query that follows the batch to its farms, and trace the result back to its evidence.
- Ask the same question through a graph-backed knowledge base and through the standard knowledge base from Part 1.

Run Part 1 first, because this notebook reuses its knowledge sources and standard knowledge base. You've already explored the ontology and graph in the graph explorer. Now you'll do the same with code.

## Step 1: Load your settings

Load the same settings as in Part 1.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]

API_VERSION = "2026-11-01-preview"
search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

## Step 2: Connect to the graph and its PDF evidence

`cocoarynth-graph` stores the business triples that the Foundry IQ plugin extracted from the PDFs ahead of time. Each triple links to the PDF chunk it came from, in `cocoarynth-chunks`. That index holds the same chunks as the four document indexes from Part 1, combined into one index, so every triple's evidence is in one place. `cocoarynth-topics` is used in Part 3.

The ontology is a checked-in OWL file, not another index. During setup, its flat class and property declarations are converted to JSON-LD and attached to `cocoarynth-graph-knowledge-source` as `graphOntology` with `kind: inline`. This November private-preview feature enables GQL-backed graph search through an ordinary search-index knowledge source.

![Diagram: cocoarynth-graph-knowledge-base uses cocoarynth-graph-knowledge-source, which searches the cocoarynth-graph index and carries the cocoarynth.owl ontology inline. The graph index links to its evidence chunks in cocoarynth-chunks by chunk_id.](../../img/diagrams/notebook-02.svg)

In [ ]:
import json
import textwrap
import time

from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

GRAPH_INDEX = "cocoarynth-graph"
CHUNKS_INDEX = "cocoarynth-chunks"
GRAPH_KNOWLEDGE_SOURCE = f"{GRAPH_INDEX}-knowledge-source"

index_client = SearchIndexClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, search_credential, api_version=API_VERSION
)
graph_client = SearchClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, GRAPH_INDEX, search_credential, api_version=API_VERSION
)
chunks_client = SearchClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, CHUNKS_INDEX, search_credential, api_version=API_VERSION
)
for name, client in ((GRAPH_INDEX, graph_client), (CHUNKS_INDEX, chunks_client)):
    print(f"{name}: {client.get_document_count()} documents")

graph_source = index_client.get_knowledge_source(GRAPH_KNOWLEDGE_SOURCE).as_dict()
inline_ontology = graph_source["searchIndexParameters"].get("graphOntology", {})
if inline_ontology.get("kind") != "inline" or not inline_ontology.get("value"):
    raise RuntimeError("The graph knowledge source needs the November inline-ontology private preview.")
print(f"Inline ontology: {inline_ontology['name']}")

## Step 3: Compare a source document with a graph assertion

Open [batch record EC26-0811](../../data/cocoarynth/documents/cocoarynth-quality/batch-record-ec26-0811.pdf). Find the batch code, bean lot, and input quantity.

### Find the batch in the documents

First, search the quality chunks in `cocoarynth-chunks` for chunks that mention the batch code. These are the same kind of chunks that the knowledge base in Part 1 retrieved.

In [ ]:
for result in chunks_client.search(
    search_text='"EC26-0811"', search_fields=["chunk"], filter="domain eq 'quality'", top=3
):
    print(f"- {result['title']} ({result['blob_path']})")
    print(textwrap.indent(textwrap.shorten(result["chunk"], 300), "    "))

### Find the batch in the graph

The graph index stores one document per **assertion**: a subject, a predicate, and an object. Because it's an ordinary search index, you can look up an assertion with a filtered search. This finds the assertion that says which bean lot batch EC26-0811 was made from. You'll query the graph with GQL in Step 5.

In [ ]:
fact = list(graph_client.search(
    search_text="*",
    filter="subject_id eq 'urn:cocoarynth:Batch:EC26-0811' and predicate eq 'madeFrom'",
    top=1,
))[0]
print(f"{fact['subject']} --{fact['predicate']}--> {fact['object']}")
print(f"chunk_id: {fact['chunk_id']}")

### Find the source of the assertion

The `chunk_id` identifies a chunk in `cocoarynth-chunks`. This `madeFrom` assertion comes from the PDF's extracted text, not the structured JSON used to generate the PDF. Check its exact evidence excerpts against the full chunk.

In [ ]:
source = chunks_client.get_document(
    fact["chunk_id"], selected_fields=["chunk_id", "title", "blob_path", "chunk"]
)
print(f"Source: {source['title']} ({source['blob_path']})")
for quote in fact["evidence_quotes"]:
    if quote not in source["chunk"]:
        raise RuntimeError("A stored evidence excerpt is not present in the PDF chunk.")
    print(f"  Exact excerpt: {quote!r}")
print(textwrap.indent(textwrap.shorten(source["chunk"], 600), "    "))

## Step 4: Break the assertion into its parts

An assertion is a **triple**: a subject, a predicate, and an object. This one connects two entities. Other assertions connect an entity to a literal value, such as a quantity or a date.

Each entity has a display name and a stable ID. The ID identifies the same entity across records. `fact_id` identifies a distinct triple; separate assertion IDs can attach that fact to different PDF chunks. The graph index's **graph profile** tells the service which index fields hold each part of the triple.

In [ ]:
for field in ("subject", "subject_type", "subject_id", "predicate", "object", "object_type", "object_id", "fact_id", "chunk_id"):
    print(f"{field:<14} {fact.get(field)}")

graph_profile = index_client.get_index(GRAPH_INDEX).as_dict()["graphProfile"]
print("\nGraph profile field mapping:")
for role, field_name in graph_profile["fields"].items():
    print(f"  {role:<22} -> {field_name}")

## Step 5: Run a one-hop GQL query

GQL (Graph Query Language) describes a pattern to match in the graph:

- `MATCH` describes the path. Parentheses are nodes, and arrows are relationships.
- `WHERE` picks a specific entity by its stable ID.
- `RETURN` chooses the columns.
- `LIMIT` caps the number of rows.

`SearchClient.graph()` runs the GQL through the SDK's native graph operation and returns typed rows, plans, statistics, and provenance. No raw HTTP fallback is needed.

This is the same batch-to-lot path you followed in the **Graph entities** tab.

In [ ]:
from azure.search.documents.models import GraphQueryProvenanceParameters


def run_graph_query(gql, plan_only=False):
    return graph_client.graph(
        gql=gql,
        execution_mode="planOnly" if plan_only else "execute",
        explain=["physicalPlan"],
        include_statistics=True,
        provenance=GraphQueryProvenanceParameters(kind="ids", max_evidence_per_row=20),
    )


GQL_ONE_HOP = """
MATCH (b:`Batch`)-[:`madeFrom`]->(l:`BeanLot`)
WHERE b.id = 'urn:cocoarynth:Batch:EC26-0811'
RETURN DISTINCT b.name AS batch, l.name AS lot, l.id AS lot_id
LIMIT 10
"""
for row in run_graph_query(GQL_ONE_HOP).value:
    print(f"{row['batch']} --madeFrom--> {row['lot']} ({row['lot_id']})")

## Step 6: Join two relationships through the same bean lot

Now extend the pattern with farms that contributed to the bean lot. Both relationships share the variable `l`, so they join on the **same** bean lot:

`(b)-[:madeFrom]->(l)<-[:contributedTo]-(f)` reads as **Batch → BeanLot ← Farm**.

In [ ]:
GQL_FARMS = """
MATCH (b:`Batch`)-[:`madeFrom`]->(l:`BeanLot`)<-[:`contributedTo`]-(f:`Farm`)
WHERE b.id = 'urn:cocoarynth:Batch:EC26-0811'
RETURN DISTINCT b.id AS batch_id, b.name AS batch,
       l.id AS lot_id, l.name AS lot,
       f.id AS farm_id, f.name AS farm
LIMIT 10
"""
print(GQL_FARMS)

## Step 7: Look at the query plan

With `executionMode` set to `planOnly`, the service plans the query without running it, so no rows come back. The plan shows how the service will look up each part of the pattern.

In [ ]:
planned = run_graph_query(GQL_FARMS, plan_only=True)
print(f"Rows returned: {len(planned.value)}")
print(json.dumps(planned.explain.as_dict() if planned.explain is not None else {}, indent=2))
print(json.dumps(planned.statistics.as_dict() if planned.statistics is not None else {}, indent=2))

## Step 8: Run the query

Run the query to find the farms behind batch EC26-0811. The statistics show how many searches the service ran and how long they took.

In [ ]:
start = time.perf_counter()
joined = run_graph_query(GQL_FARMS)
graph_query_ms = (time.perf_counter() - start) * 1000
paths = joined.value

for row in paths:
    print(f"{row['batch']} --madeFrom--> {row['lot']} <--contributedTo-- {row['farm']}")
print(json.dumps(joined.statistics.as_dict() if joined.statistics is not None else {}, indent=2))

## Step 9: Follow each path back to its evidence

Each returned row includes **provenance**: the IDs of the assertions that support it and the source chunks they came from. Follow those IDs to read the actual evidence.

The evidence count includes a relation: `exact` or `lowerBound`. Read that relation rather than assuming the bounded evidence collection contains every assertion, or every record that might mention the same fact.

In [ ]:
for row in paths:
    provenance = row.provenance
    if provenance is None or not provenance.evidence:
        raise RuntimeError("The graph query did not return the requested assertion evidence.")
    print(f"{row['farm']} (evidence count: {provenance.evidence_count.as_dict()})")
    for item in provenance.evidence:
        assertion = graph_client.get_document(item.edge_id)
        source = chunks_client.get_document(
            item.chunk_id, selected_fields=["title", "blob_path", "chunk"]
        )
        print(f"  {assertion['subject']} --{assertion['predicate']}--> {assertion['object']}")
        print(f"    PDF source: {source['blob_path']}")
        print(textwrap.indent(textwrap.shorten(source["chunk"], 200), "      "))

## Step 10: Ask a graph-backed knowledge base

You wrote the GQL above by hand. Now create a knowledge base over the graph knowledge source and ask the question in natural language. The knowledge base can use a **graph search** tool that follows relationships in the graph, as well as ordinary text search.

The retrieval instructions encourage the model to use graph search, but they don't force it. The next cell checks the activity to confirm that graph search actually ran.

In [ ]:
from IPython.display import Markdown, display

from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeBaseMicrosoftRetriever1Model,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
    SearchIndexKnowledgeSourceParams,
)

GRAPH_KB_NAME = "cocoarynth-graph-knowledge-base"
graph_knowledge_base = KnowledgeBase(
    name=GRAPH_KB_NAME,
    description="Relationship reasoning over the Cocoarynth graph knowledge source",
    knowledge_sources=[KnowledgeSourceReference(name=GRAPH_KNOWLEDGE_SOURCE)],
    models=[
        KnowledgeBaseMicrosoftRetriever1Model(),
        KnowledgeBaseAzureOpenAIModel(
            azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                resource_url=AZURE_OPENAI_ENDPOINT,
                deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
                model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
                api_key=AZURE_OPENAI_KEY,
            )
        )
    ],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions=(
        "For relationship questions use the graph_search tool and the supplied ontology. "
        "For batch traceability follow Batch-madeFrom->BeanLot<-contributedTo-Farm. "
        "Use stable IDs when provided."
    ),
    answer_instructions="Answer only from returned evidence, cite reference IDs, and say when evidence is missing.",
)
index_client.create_or_update_knowledge_base(graph_knowledge_base)
print(f"Knowledge base '{GRAPH_KB_NAME}' created or updated.\n")

RELATIONSHIP_QUESTION = "Which farms contributed cacao to the bean lot used in batch EC26-0811? Give the farm names."


def relationship_request(knowledge_source_names):
    return KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=RELATIONSHIP_QUESTION)])
        ],
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
        include_activity=True,
        knowledge_source_params=[
            SearchIndexKnowledgeSourceParams(
                knowledge_source_name=name,
                include_references=True,
                include_reference_source_data=True,
                always_query_source=True,
            )
            for name in knowledge_source_names
        ],
    )


graph_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=GRAPH_KB_NAME,
    credential=search_credential,
    api_version=API_VERSION,
)
start = time.perf_counter()
graph_result = graph_kb_client.retrieve(relationship_request([GRAPH_KNOWLEDGE_SOURCE]))
graph_kb_ms = (time.perf_counter() - start) * 1000
display(Markdown(graph_result.response[0].content[0].text))

print("Activity:")
for activity in graph_result.activity:
    step = activity.as_dict()
    print(f"  {step['type']:<24} {step.get('knowledgeSourceName', ''):<32} results: {step.get('count', '-')}")

print("\nReferences:")
for reference in graph_result.references:
    source_data = reference.source_data or {}
    print(f"  [{reference.id}] {source_data.get('title', reference.type)}")
    if "chunk" in source_data:
        print(textwrap.indent(textwrap.shorten(source_data["chunk"], 200), "      "))

### Check that graph search actually ran

Look for a `graphSearch` step in the activity that succeeded and returned results. The cell below shows the request that the planning model sent to graph search, and the rows that graph search returned. The service turns that request into GQL behind the scenes, but it doesn't return the query itself, unlike the GQL you wrote by hand earlier in this notebook.

In [ ]:
graph_steps = [
    step.as_dict() for step in graph_result.activity or []
    if step.as_dict().get("type") == "graphSearch"
]
if not graph_steps:
    raise RuntimeError("No graph search ran. Text search alone does not demonstrate the private-preview graph feature.")
for step in graph_steps:
    arguments = step.get("graphSearchArguments") or {}
    print(f"graphSearch: outcome={arguments.get('outcome')}, results={step.get('count')}, {step.get('elapsedMs')} ms")
    print(f"Request: {arguments.get('naturalLanguageQuery')}\n")

for reference in graph_result.references or []:
    graph_rows = (reference.source_data or {}).get("rows", [])
    for row in graph_rows:
        # Skip the gqlref columns, which identify each node and edge for provenance
        print({column: value for column, value in row.items() if not str(value).startswith("gqlref:")})

if not any(
    (step.get("graphSearchArguments") or {}).get("outcome", "").lower() == "success"
    and not step.get("error") and (step.get("count") or 0) > 0
    for step in graph_steps
):
    raise RuntimeError("Graph search did not succeed with results; inspect the activity before comparing answers.")

## Step 11: Compare with ordinary document retrieval

Ask the same question using the standard knowledge base from Part 1, which searches the original documents.

Ordinary retrieval may answer correctly too. What the graph adds is an explicit, inspectable path between entities, with provenance for each link. It doesn't guarantee a better answer or a faster one.

In [ ]:
STANDARD_KB_NAME = "cocoarynth-standard-knowledge-base"
document_knowledge_sources = [
    f"cocoarynth-{domain}-knowledge-source" for domain in ("production", "sales", "policies")
]

standard_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=STANDARD_KB_NAME,
    credential=search_credential,
    api_version=API_VERSION,
)
start = time.perf_counter()
document_result = standard_client.retrieve(relationship_request(document_knowledge_sources))
document_kb_ms = (time.perf_counter() - start) * 1000
display(Markdown(document_result.response[0].content[0].text))

print("Activity:")
for activity in document_result.activity:
    step = activity.as_dict()
    print(f"  {step['type']:<24} {step.get('knowledgeSourceName', ''):<32} results: {step.get('count', '-')}")

print("\nReferences:")
for reference in document_result.references:
    source_data = reference.source_data or {}
    print(f"  [{reference.id}] {source_data.get('title', reference.type)}")
    if "chunk" in source_data:
        print(textwrap.indent(textwrap.shorten(source_data["chunk"], 200), "      "))

print("\nResponse times (one request each):")
print(f"  Handwritten GQL query:       {graph_query_ms:>6.0f} ms  (graph traversal only, no model)")
print(f"  Graph knowledge base:        {graph_kb_ms:>6.0f} ms  (model planning, graph search, answer)")
print(f"  Standard knowledge base:     {document_kb_ms:>6.0f} ms  (model planning, document search, answer)")

## ✅ Mission complete

**What you did:**

* ✓ **Assertions and evidence**: Matched a stored triple to the PDF chunk it came from.
* ✓ **GQL**: Followed batch EC26-0811 to its farms and traced each link back to its evidence.
* ✓ **Graph knowledge base**: Asked the same question through graph-backed retrieval and compared it with the standard knowledge base.

A graph makes relationship questions inspectable: typed entities, stable IDs, explicit joins, and provenance for each link. Natural-language graph retrieval can take longer than a handwritten query because the model plans the search. Stored relationships can also be incomplete, so check the original records before relying on them.

For a full customer reply, also check [the data-sharing policy](../../data/cocoarynth/documents/cocoarynth-policies/data-sharing-policy.pdf). Knowing which farm grew the cacao doesn't mean you can share personal details about the people there.

**Try it yourself:** Trace the July complaint to its depot stop and storage site. Use the **Graph ontology** tab in the graph explorer app to find the relationships to follow.

➡️ Return to lab instructions.